In [ ]:
# KAGGLE BENCHMARK + FASTAPI — ONE-CELL SCRIPT
# Attach Models and a Kaggle Dataset using Add Input. Then run this cell.
# Models: .joblib / .pt only. Select DATASET ATTACHMENT, not CSV files.
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch
import ipywidgets as w
from IPython.display import display, clear_output
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

ROOT = Path('/kaggle/input')
OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
MODELS = {}
BAD = ('preprocessor', 'scaler', 'encoder', 'imputer', 'tokenizer')

def model_files():
    return sorted((p for p in ROOT.rglob('*') if p.is_file()
                   and p.suffix.lower() in {'.pt', '.joblib'}
                   and not any(word in p.stem.lower() for word in BAD)), key=str)

def dataset_folders():
    # Kaggle datasets can mount at /kaggle/input/<slug> or
    # /kaggle/input/datasets/<owner>/<slug>.
    found = []
    if not ROOT.exists():
        return found
    for p in ROOT.rglob('*.csv'):
        relative = p.relative_to(ROOT).parts
        if not relative or relative[0].lower() in {'models', 'notebooks'}:
            continue
        if relative[0].lower() == 'datasets' and len(relative) >= 4:
            folder = ROOT.joinpath(*relative[:3])
        elif relative[0].lower() == 'datasets' and len(relative) >= 3:
            folder = ROOT.joinpath(*relative[:2])
        else:
            folder = ROOT / relative[0]
        if folder not in found:
            found.append(folder)
    return sorted(found, key=str)

def csv_in_dataset(folder):
    choices = sorted(Path(folder).rglob('*.csv'), key=str)
    if not choices:
        raise ValueError('This Kaggle Dataset has no CSV table.')
    # Dataset selection is the only choice. Prefer obvious evaluation data.
    ranking = lambda p: (0 if any(s in p.stem.lower() for s in ('test', 'eval', 'valid')) else 1, str(p))
    return sorted(choices, key=ranking)[0]

models_ui = w.SelectMultiple(description='Models', rows=7, layout=w.Layout(width='98%'))
dataset_ui = w.Dropdown(description='Dataset', layout=w.Layout(width='98%'))
target_ui = w.Dropdown(description='Target', layout=w.Layout(width='75%'))
task_ui = w.Dropdown(description='Task', options=['classification', 'regression'])
refresh_btn = w.Button(description='Refresh inputs')
load_btn = w.Button(description='Load models', button_style='success')
check_btn = w.Button(description='Check inputs')
bench_btn = w.Button(description='Benchmark', button_style='primary')
query_btn = w.Button(description='Query FastAPI')
log = w.Output()

def selected_csv():
    if not dataset_ui.value:
        raise ValueError('Attach a Kaggle Dataset using Add Input > Datasets.')
    return csv_in_dataset(dataset_ui.value)

def update_target(change=None):
    old = target_ui.value
    try:
        columns = pd.read_csv(selected_csv(), nrows=0).columns.tolist()
        target_ui.options = columns
        if columns:
            target_ui.value = old if old in columns else columns[-1]
    except Exception:
        target_ui.options = []

def refresh(change=None):
    old_models = set(models_ui.value)
    old_dataset = dataset_ui.value
    ms, ds = model_files(), dataset_folders()
    models_ui.options = [(str(p), str(p)) for p in ms]
    models_ui.value = tuple(p for p in old_models if p in {str(x) for x in ms})
    dataset_ui.options = [(p.name + ' | ' + str(p), str(p)) for p in ds]
    if old_dataset in {str(x) for x in ds}:
        dataset_ui.value = old_dataset
    update_target()
    with log:
        clear_output(wait=True)
        print(f'{len(ms)} model file(s); {len(ds)} Kaggle dataset attachment(s).')
        if ds:
            p = selected_csv()
            print('Selected dataset:', dataset_ui.value)
            print('CSV used automatically:', p)
            if len(list(Path(dataset_ui.value).rglob('*.csv'))) > 1:
                print('Multiple CSVs exist in this dataset; selected one automatically.')
        else:
            print('No dataset attached. Kaggle > Add Input > Datasets.')

def load_one(path):
    if Path(path).suffix.lower() == '.joblib':
        obj = joblib.load(path)  # Only load trusted files.
        if not callable(getattr(obj, 'predict', None)):
            raise ValueError('File is not a trained estimator (possibly a preprocessor).')
        return obj, 'estimator'
    try:
        obj = torch.jit.load(path, map_location='cpu')
        obj.eval()
        return obj, 'torchscript'
    except Exception:
        ckpt = torch.load(path, map_location='cpu', weights_only=True)
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            raise ValueError('PyTorch state_dict checkpoint needs its original nn.Module class; '
                             f"dimensions={ckpt.get('dimensions')}. Not TorchScript.")
        raise ValueError('Unsupported PyTorch .pt format')

def load(change=None):
    MODELS.clear()
    with log:
        clear_output(wait=True)
        for path in models_ui.value:
            try:
                obj, kind = load_one(path)
                name = Path(path).stem
                i = 2
                while name in MODELS:
                    name = f'{Path(path).stem}_{i}'
                    i += 1
                MODELS[name] = {'obj': obj, 'kind': kind, 'path': path}
                print('LOADED:', name, kind)
            except Exception as e:
                print('NOT LOADED:', path, '|', e)
        print('Inference models ready:', len(MODELS))

def data():
    path = selected_csv()
    df = pd.read_csv(path)
    target = target_ui.value
    if not target or target not in df.columns:
        raise ValueError('Select a valid target column.')
    df = df.dropna(subset=[target])
    if df.empty:
        raise ValueError('Dataset contains no labeled records.')
    return df.drop(columns=[target]), df[target]

def predict(name, X):
    item = MODELS[name]
    obj = item['obj']
    if item['kind'] == 'estimator':
        if hasattr(obj, 'feature_names_in_'):
            expected = list(obj.feature_names_in_)
            missing = [col for col in expected if col not in X.columns]
            if missing:
                raise ValueError(f'Missing model inputs: {missing}')
            X = X.loc[:, expected]
        return np.asarray(obj.predict(X)).reshape(-1)
    with torch.no_grad():
        result = obj(torch.tensor(X.to_numpy(dtype=np.float32)))
        if isinstance(result, (tuple, list)):
            result = result[0]
        raw = result.detach().cpu().numpy()
    if task_ui.value == 'regression':
        return raw.reshape(-1)
    if raw.ndim == 2 and raw.shape[1] > 1:
        return raw.argmax(axis=1)
    return (raw.reshape(-1) >= .5).astype(int)  # Assumes binary probabilities.

def check(change=None):
    with log:
        clear_output(wait=True)
        try:
            X, y = data()
            print('Dataset attachment:', dataset_ui.value)
            print('CSV:', selected_csv(), '| target:', target_ui.value, '| rows:', len(X))
            print('Features:', list(X.columns))
            for name in MODELS:
                try:
                    print('PASS:', name, predict(name, X.iloc[:1]).tolist())
                except Exception as e:
                    print('FAIL:', name, e)
            if not MODELS:
                print('No inference models loaded.')
        except Exception as e:
            print('INPUT ERROR:', e)

def benchmark():
    if not MODELS:
        raise ValueError('Load a trained model first.')
    X, y = data()
    rows = []
    for name, item in MODELS.items():
        try:
            predict(name, X.iloc[:min(2, len(X))])
            times = []
            for _ in range(5):
                start = time.perf_counter()
                pred = predict(name, X)
                times.append((time.perf_counter() - start) * 1000)
            row = {'model': name, 'framework': item['kind'], 'rows': len(X),
                   'median_ms': round(float(np.median(times)), 3)}
            if task_ui.value == 'classification':
                row.update(accuracy=accuracy_score(y, pred),
                           f1=f1_score(y, pred, average='weighted', zero_division=0))
            else:
                row.update(rmse=float(np.sqrt(mean_squared_error(y, pred))),
                           r2=r2_score(y, pred))
            rows.append(row)
        except Exception as e:
            rows.append({'model': name, 'error': str(e)})
    board = pd.DataFrame(rows)
    metric = 'f1' if task_ui.value == 'classification' else 'rmse'
    if metric in board.columns:
        board = board.sort_values(metric, ascending=(metric == 'rmse'), na_position='last')
    board.to_csv(OUT / 'benchmark_leaderboard.csv', index=False)
    return board

app = FastAPI(title='Kaggle Dataset Benchmark and Serving Lab')
class Request(BaseModel):
    instances: list[dict]

@app.get('/health')
def health():
    return {'ok': True, 'models_loaded': len(MODELS)}

@app.get('/models')
def model_list():
    return [{'name': name, 'kind': item['kind']} for name, item in MODELS.items()]

@app.post('/predict/{name}')
def predict_one(name: str, request: Request):
    if name not in MODELS:
        raise HTTPException(404, 'Unknown model')
    try:
        return {'model': name, 'predictions': predict(name, pd.DataFrame(request.instances)).tolist()}
    except Exception as e:
        raise HTTPException(422, str(e))

@app.post('/predict-all')
def predict_all(request: Request):
    X = pd.DataFrame(request.instances)
    results = {}
    for name in MODELS:
        try:
            results[name] = predict(name, X).tolist()
        except Exception as e:
            results[name] = {'error': str(e)}
    return {'results': results}

client = TestClient(app)

def bench_click(change=None):
    with log:
        clear_output(wait=True)
        try:
            display(benchmark())
            print('Saved:', OUT / 'benchmark_leaderboard.csv')
        except Exception as e:
            print('BENCHMARK ERROR:', e)

def query(change=None):
    with log:
        clear_output(wait=True)
        try:
            X, _ = data()
            payload = {'instances': json.loads(X.iloc[:1].to_json(orient='records'))}
            print('GET /health:', client.get('/health').json())
            print('GET /models:', client.get('/models').json())
            response = client.post('/predict-all', json=payload)
            print('POST /predict-all:', response.status_code)
            print(json.dumps(response.json(), indent=2))
        except Exception as e:
            print('API ERROR:', e)

# Create and render each dropdown ONCE.
dataset_ui.observe(update_target, names='value')
refresh_btn.on_click(refresh)
load_btn.on_click(load)
check_btn.on_click(check)
bench_btn.on_click(bench_click)
query_btn.on_click(query)
display(w.HTML('<h3>Kaggle Dataset + Model Benchmark / FastAPI Lab</h3>'))
display(models_ui, dataset_ui, target_ui, task_ui)
display(w.HBox([refresh_btn, load_btn, check_btn, bench_btn, query_btn]), log)
refresh()
print('Ready: select a Kaggle dataset attachment, then load models and test.')
# END OF ENTIRE ONE-CELL SCRIPT
